# <center> CUDA PROGRAMMING LABORATORY</center>

# Accessing the CUDA Laboratory Environment Using Kaggle

## Objective

This laboratory uses **Kaggle Notebooks** as the development environment for CUDA programming. Each student must create a Kaggle account, complete mobile verification, and enable GPU acceleration before performing the laboratory experiments.

---


## Step 1: Create a Kaggle Account

1. Open your web browser.
2. Visit **https://www.kaggle.com**.
3. Click **Register** (or **Sign Up**).
4. Create a Kaggle account using one of the following options:
   - Google Account
   - Microsoft Account
   - Email Address
5. Complete the registration process and log in to your account.

---

## Step 2: Complete Mobile Verification

> **Important:** Mobile verification is required to access GPU resources in Kaggle.

1. Log in to your Kaggle account.
2. Click on your **Profile** icon.
3. Select **Settings**.
4. Navigate to the **Phone Verification** section.
5. Enter your mobile number.
6. Click **Send Verification Code**.
7. Enter the One-Time Password (OTP) received on your mobile phone.
8. Click **Verify**.

After successful verification, your account becomes eligible to use GPU resources.

---

## Step 3: Create a New Notebook

1. Click the **Create** button.
2. Select **New Notebook**.
3. Wait until the notebook opens.

---

## Step 4: Enable GPU Accelerator

1. Open the **Notebook Settings** panel.
2. Locate the **Accelerator** option.
3. Select **GPU** from the available options.
4. Save the notebook settings.
5. Wait a few seconds while Kaggle allocates a GPU.

---

## Step 5: Verify GPU Availability

Run the following command in a notebook cell.

```python
!nvidia-smi
```

### What does this command do?

The `!nvidia-smi` (NVIDIA System Management Interface) command displays information about the NVIDIA GPU assigned to your Kaggle notebook.

It provides details such as:

- GPU model (e.g., NVIDIA Tesla T4, P100, or L4)
- Installed CUDA version
- GPU driver version
- GPU memory capacity
- Current GPU utilization
- Running GPU processes

### Expected Output

If the GPU is enabled successfully, the output will be similar to the following:

```text
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 550.xx.xx      Driver Version: 550.xx.xx     CUDA Version: 12.x |
| GPU  Name                 Memory-Usage       GPU-Util                       |
|  0   Tesla T4             0MiB / 15360MiB        0%                         |
+-----------------------------------------------------------------------------+
```


---

# <center>EXPERIMENT 1</center>

# <center>VECTOR ADDITION</center>

---


# Vector Addition

Vector addition is the process of adding two vectors by combining their corresponding elements to produce a new vector. In CUDA programming, vector addition is a fundamental example used to demonstrate parallel computing, where each GPU thread is responsible for adding one pair of elements from the input vectors simultaneously.

Mathematically, vector addition is represented as:

\[
C[i] = A[i] + B[i],   i = 0, 1, 2, N-1
\]

where:

- **A** = First input vector
- **B** = Second input vector
- **C** = Resultant vector
- **N** = Number of elements in each vector

### Example

```
Vector A : [1, 2, 3, 4]

Vector B : [5, 6, 7, 8]

Vector C : [6, 8, 10, 12]
```

In CUDA, each element of the resultant vector is computed independently by a separate GPU thread, making vector addition an ideal example for understanding data parallelism.

## Program 1.1: Basic Implementation

This program demonstrates the **basic implementation of vector addition using CUDA**. It focuses only on the essential steps required to execute a CUDA program, making it suitable for beginners who are learning CUDA programming for the first time.

The program performs the following operations:

- Defines a CUDA kernel using the `__global__` keyword.
- Allocates memory on the GPU.
- Copies input vectors from the CPU to the GPU.
- Executes the kernel to perform vector addition.
- Copies the result back to the CPU.
- Displays only the final output vector.
- Releases the allocated GPU memory.

This implementation helps students understand the fundamental workflow of CUDA programming without introducing additional concepts or debugging information.

In [13]:

%%writefile vector.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

// Function Prototype
__global__ void vectorAdd(int *A, int *B, int *C);

int main()
{
    // Host Arrays
    int h_A[N] = {1,2,3,4,5,6,7,8};
    int h_B[N] = {10,20,30,40,50,60,70,80};
    int h_C[N];

    // Device Pointers
    int *d_A, *d_B, *d_C;

    // Allocate GPU Memory
    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));
    cudaMalloc((void**)&d_C, N * sizeof(int));

    // Copy Data to GPU
    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(int), cudaMemcpyHostToDevice);

    // Launch Kernel
    vectorAdd<<<1, N>>>(d_A, d_B, d_C);

    // Wait for GPU
    cudaDeviceSynchronize();

    // Copy Result Back
    cudaMemcpy(h_C, d_C, N * sizeof(int), cudaMemcpyDeviceToHost);

    // Display Result
    printf("Result:\n");

    for(int i = 0; i < N; i++)
    {
        printf("%d ", h_C[i]);
    }

    printf("\n");

    // Free GPU Memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

// CUDA Kernel
__global__ void vectorAdd(int *A, int *B, int *C)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if(idx < N)
    {
        C[idx] = A[idx] + B[idx];
    }
}

Overwriting vector.cu


In [16]:
!nvcc vector.cu -o vector

/bin/bash: line 1: nvcc: command not found


In [12]:
!./vector

/bin/bash: line 1: ./vector: No such file or directory


## Program 1.2: Detailed Implementation

This program presents a **detailed implementation of vector addition using CUDA**. In addition to performing vector addition, it displays the execution of each major step involved in CUDA programming. This version is intended for educational purposes and helps students understand how a CUDA program executes on both the CPU (Host) and GPU (Device).

The program demonstrates the following concepts:

- Declaration of a CUDA kernel using the `__global__` keyword.
- Allocation of memory on the GPU using `cudaMalloc()`.
- Data transfer between the CPU and GPU using `cudaMemcpy()`.
- Kernel launch configuration using `<<<Grid, Block>>>`.
- Calculation of the global thread index.
- Parallel execution of GPU threads.
- Display of the computation performed by each GPU thread.
- Synchronization of GPU execution using `cudaDeviceSynchronize()`.
- Copying the computed results back to the CPU.
- Releasing GPU memory using `cudaFree()`.

This implementation enables students to observe the complete execution flow of a CUDA application and serves as a foundation for developing more advanced parallel programming applications.

In [ ]:
%%writefile detail_vector.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

//=========================================================
// Function Prototype
//=========================================================
__global__ void vectorAdd(int *A, int *B, int *C);

//=========================================================
// Main Function
//=========================================================
int main()
{
    // Host Arrays
    int h_A[N] = {1,2,3,4,5,6,7,8};
    int h_B[N] = {10,20,30,40,50,60,70,80};
    int h_C[N];

    // Device Pointers
    int *d_A;
    int *d_B;
    int *d_C;

    printf("\n=========================================\n");
    printf("      VECTOR ADDITION USING CUDA\n");
    printf("=========================================\n\n");

    // Display Input Vector A
    printf("Input Vector A:\n");
    for(int i = 0; i < N; i++)
        printf("%d ", h_A[i]);

    // Display Input Vector B
    printf("\n\nInput Vector B:\n");
    for(int i = 0; i < N; i++)
        printf("%d ", h_B[i]);

    // Allocate GPU Memory
    printf("\n\nAllocating GPU Memory...\n");

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));
    cudaMalloc((void**)&d_C, N * sizeof(int));

    printf("GPU Memory Allocated Successfully.\n");

    // Copy Data from CPU to GPU
    printf("\nCopying Data from CPU to GPU...\n");

    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(int), cudaMemcpyHostToDevice);

    printf("Data Transfer Completed.\n");

    // Launch CUDA Kernel
    printf("\nLaunching CUDA Kernel...\n\n");

    vectorAdd<<<1, N>>>(d_A, d_B, d_C);

    // Wait for GPU to Finish
    cudaDeviceSynchronize();

    printf("\nKernel Execution Completed.\n");

    // Copy Result Back to CPU
    printf("\nCopying Result from GPU to CPU...\n");

    cudaMemcpy(h_C, d_C, N * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Copy Completed.\n");

    // Display Result
    printf("\nFinal Result Vector:\n");

    for(int i = 0; i < N; i++)
        printf("C[%d] = %d\n", i, h_C[i]);

    // Free GPU Memory
    printf("\nFreeing GPU Memory...\n");

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    printf("GPU Memory Released Successfully.\n");

    printf("\nProgram Executed Successfully.\n");

    return 0;
}

//=========================================================
// CUDA Kernel Function
//=========================================================
__global__ void vectorAdd(int *A, int *B, int *C)
{
    // Calculate Global Thread Index
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Perform Vector Addition
    if(idx < N)
    {
        C[idx] = A[idx] + B[idx];

        printf("GPU Thread %d : %d + %d = %d\n",
               idx,
               A[idx],
               B[idx],
               C[idx]);
    }
}

Overwriting detail_vector.cu


In [ ]:
!nvcc detail_vector.cu -o detail_vector

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./detail_vector


      VECTOR ADDITION USING CUDA

Input Vector A:
1 2 3 4 5 6 7 8 

Input Vector B:
10 20 30 40 50 60 70 80 

Allocating GPU Memory...
GPU Memory Allocated Successfully.

Copying Data from CPU to GPU...
Data Transfer Completed.

Launching CUDA Kernel...

GPU Thread 0 : 1 + 10 = 11
GPU Thread 1 : 2 + 20 = 22
GPU Thread 2 : 3 + 30 = 33
GPU Thread 3 : 4 + 40 = 44
GPU Thread 4 : 5 + 50 = 55
GPU Thread 5 : 6 + 60 = 66
GPU Thread 6 : 7 + 70 = 77
GPU Thread 7 : 8 + 80 = 88

Kernel Execution Completed.

Copying Result from GPU to CPU...
Copy Completed.

Final Result Vector:
C[0] = 11
C[1] = 22
C[2] = 33
C[3] = 44
C[4] = 55
C[5] = 66
C[6] = 77
C[7] = 88

Freeing GPU Memory...
GPU Memory Released Successfully.

Program Executed Successfully.


## Program Outcome

After successfully completing this experiment, You will be able to:

1. Understand the concept of vector addition using parallel computing.
2. Develop and execute a basic CUDA program using the `__global__` kernel function.
3. Allocate and deallocate memory on the GPU using `cudaMalloc()` and `cudaFree()`.
4. Transfer data between the Host (CPU) and Device (GPU) using `cudaMemcpy()`.
5. Launch a CUDA kernel using the execution configuration `<<<Grid, Block>>>`.
6. Understand the execution of multiple GPU threads for parallel computation.
7. Synchronize CPU and GPU execution using `cudaDeviceSynchronize()`.
8. Retrieve and verify computation results from the GPU.
9. Differentiate between Host memory and Device memory.
10. Explain the complete execution flow of a simple CUDA application.

---
<div align="center">


## EXPERIMENT 2

# PARALLEL REDUCTION

</div>

---

## Parallel Reduction

**Parallel Reduction** is a parallel computing technique used to combine multiple elements of an array into a **single result** by repeatedly applying an operation such as **addition, multiplication, minimum, or maximum**. In CUDA, multiple GPU threads work simultaneously to perform the reduction, making the computation significantly faster than a sequential CPU implementation for large datasets.

The most common example of parallel reduction is **finding the sum of all elements in an array**. Instead of adding elements one by one, pairs of elements are added in parallel, and the intermediate results are repeatedly combined until only a single value remains.


In CUDA, these additions are performed **simultaneously by multiple GPU threads**, greatly reducing the execution time for large arrays. Parallel reduction is a fundamental algorithm used in many GPU applications, including matrix operations, image processing, machine learning, scientific computing, histogram computation, and prefix sum (scan) algorithms.

## Program 2.1: Basic Implementation

This program demonstrates the **basic implementation of Parallel Reduction using CUDA**. It computes the **sum of all elements in an array** by performing the reduction operation in parallel on the GPU. The program focuses on the essential CUDA concepts required to implement a reduction algorithm without displaying intermediate execution details.

The program performs the following operations:

- Declares a CUDA kernel using the `__global__` keyword.
- Allocates memory on the GPU using `cudaMalloc()`.
- Copies the input array from the Host (CPU) to the Device (GPU).
- Loads input data into Shared Memory (`__shared__`) for faster access.
- Performs parallel reduction using multiple GPU threads.
- Synchronizes threads using `__syncthreads()` after each reduction step.
- Stores the final reduced value in global memory.
- Copies the final result from the GPU back to the CPU.
- Displays only the final sum of all array elements.
- Releases the allocated GPU memory using `cudaFree()`.

This implementation helps students understand the basic workflow of a CUDA-based parallel reduction algorithm and introduces the concepts of **Shared Memory**, **Thread Synchronization**, and **Parallel Summation**, which are fundamental to many GPU computing applications.

In [ ]:
%%writefile reduction.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

//=========================================================
// Function Prototype
//=========================================================
__global__ void parallelReduction(int *input, int *output);

//=========================================================
// Main Function
//=========================================================
int main()
{
    // Host Arrays
    int h_input[N] = {1,2,3,4,5,6,7,8};
    int h_output;

    // Device Pointers
    int *d_input;
    int *d_output;

    // Allocate GPU Memory
    cudaMalloc((void**)&d_input, N * sizeof(int));
    cudaMalloc((void**)&d_output, sizeof(int));

    // Copy Input Data to GPU
    cudaMemcpy(d_input, h_input, N * sizeof(int), cudaMemcpyHostToDevice);

    // Launch Kernel
    parallelReduction<<<1, N>>>(d_input, d_output);

    // Wait for GPU to Finish
    cudaDeviceSynchronize();

    // Copy Result Back to CPU
    cudaMemcpy(&h_output, d_output, sizeof(int), cudaMemcpyDeviceToHost);

    // Display Result
    printf("Sum = %d\n", h_output);

    // Free GPU Memory
    cudaFree(d_input);
    cudaFree(d_output);

    return 0;
}

//=========================================================
// CUDA Kernel Function
//=========================================================
__global__ void parallelReduction(int *input, int *output)
{
    __shared__ int data[N];

    int tid = threadIdx.x;

    // Copy data from global memory to shared memory
    data[tid] = input[tid];

    __syncthreads();

    // Parallel Reduction
    for(int stride = N / 2; stride > 0; stride /= 2)
    {
        if(tid < stride)
        {
            data[tid] += data[tid + stride];
        }

        __syncthreads();
    }

    // Store Final Result
    if(tid == 0)
    {
        output[0] = data[0];
    }
}

Overwriting reduction.cu


In [ ]:
!nvcc reduction.cu -o reduction

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./reduction

Sum = 36


## Program 2.2: Detailed Implementation

This program presents a **detailed implementation of Parallel Reduction using CUDA**. Unlike the basic implementation, this version displays the execution of every major step involved in the reduction process. It is designed for educational purposes to help students understand how multiple GPU threads cooperate to reduce an array into a single value.

The program demonstrates the following concepts:

- Declaration of a CUDA kernel using the `__global__` keyword.
- Allocation of GPU memory using `cudaMalloc()`.
- Transfer of data between the Host (CPU) and Device (GPU) using `cudaMemcpy()`.
- Loading input data from Global Memory into Shared Memory (`__shared__`) for faster access.
- Execution of the reduction algorithm using multiple GPU threads.
- Synchronization of threads using `__syncthreads()` after each reduction step.
- Display of thread-wise computations performed during each reduction stage.
- Observation of how the number of active threads decreases after every iteration.
- Storage of the final reduced value in global memory.
- Copying the computed result from the GPU back to the CPU.
- Releasing GPU memory using `cudaFree()`.

During execution, the program prints:

- The input array.
- GPU memory allocation status.
- Data transfer between the CPU and GPU.
- Values loaded into Shared Memory by each thread.
- Reduction operations performed by each active thread.
- Intermediate results after every reduction stage.
- The final sum computed on the GPU.
- Result transfer back to the CPU.
- GPU memory deallocation status.

This implementation enables students to visualize the complete execution flow of the **Parallel Reduction** algorithm and understand how **Shared Memory**, **Thread Synchronization**, and **Parallel Execution** work together to efficiently compute the sum of an array. It also provides a strong foundation for advanced CUDA algorithms such as **Prefix Sum (Scan)**, **Histogram Computation**, **Matrix Operations**, and **Parallel Sorting**.

In [ ]:
%%writefile details_reduction.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

//=========================================================
// Function Prototype
//=========================================================
__global__ void parallelReduction(int *input, int *output);

//=========================================================
// Main Function
//=========================================================
int main()
{
    // Host Arrays
    int h_input[N] = {1,2,3,4,5,6,7,8};
    int h_output;

    // Device Pointers
    int *d_input;
    int *d_output;

    printf("\n=============================================\n");
    printf("      PARALLEL REDUCTION USING CUDA\n");
    printf("=============================================\n");

    // Display Input
    printf("\nInput Array:\n");
    for(int i=0;i<N;i++)
        printf("%d ", h_input[i]);

    printf("\n");

    // Allocate GPU Memory
    printf("\nAllocating GPU Memory...\n");

    cudaMalloc((void**)&d_input, N*sizeof(int));
    cudaMalloc((void**)&d_output, sizeof(int));

    printf("GPU Memory Allocated Successfully.\n");

    // Copy Data
    printf("\nCopying Input Array from CPU to GPU...\n");

    cudaMemcpy(d_input, h_input, N*sizeof(int), cudaMemcpyHostToDevice);

    printf("Data Transfer Completed.\n");

    // Launch Kernel
    printf("\nLaunching CUDA Kernel...\n\n");

    parallelReduction<<<1,N>>>(d_input,d_output);

    cudaDeviceSynchronize();

    printf("\nKernel Execution Completed.\n");

    // Copy Result
    printf("\nCopying Result from GPU to CPU...\n");

    cudaMemcpy(&h_output,d_output,sizeof(int),cudaMemcpyDeviceToHost);

    printf("Copy Completed.\n");

    // Display Result
    printf("\n=============================================\n");
    printf("Final Sum = %d\n", h_output);
    printf("=============================================\n");

    // Free Memory
    printf("\nFreeing GPU Memory...\n");

    cudaFree(d_input);
    cudaFree(d_output);

    printf("GPU Memory Released Successfully.\n");

    printf("\nProgram Executed Successfully.\n");

    return 0;
}

//=========================================================
// CUDA Kernel Function
//=========================================================
__global__ void parallelReduction(int *input, int *output)
{
    __shared__ int data[N];

    int tid = threadIdx.x;

    // Copy into Shared Memory
    data[tid] = input[tid];

    printf("Thread %d copied %d into Shared Memory\n",
           tid,
           data[tid]);

    __syncthreads();

    // Reduction Process
    for(int stride=N/2; stride>0; stride/=2)
    {
        if(tid < stride)
        {
            printf("\nStride %d\n", stride);

            printf("Thread %d : %d + %d = ",
                    tid,
                    data[tid],
                    data[tid+stride]);

            data[tid] += data[tid+stride];

            printf("%d\n", data[tid]);
        }

        __syncthreads();

        if(tid==0)
        {
            printf("\nShared Memory after Stride %d : ", stride);

            for(int i=0;i<stride;i++)
                printf("%d ", data[i]);

            printf("\n------------------------------------\n");
        }

        __syncthreads();
    }

    if(tid==0)
    {
        output[0]=data[0];

        printf("\nReduction Completed.\n");
        printf("Final Sum Stored = %d\n", data[0]);
    }
}

Overwriting details_reduction.cu


In [ ]:
!nvcc details_reduction.cu -o details_reduction

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./details_reduction


      PARALLEL REDUCTION USING CUDA

Input Array:
1 2 3 4 5 6 7 8 

Allocating GPU Memory...
GPU Memory Allocated Successfully.

Copying Input Array from CPU to GPU...
Data Transfer Completed.

Launching CUDA Kernel...

Thread 0 copied 1 into Shared Memory
Thread 1 copied 2 into Shared Memory
Thread 2 copied 3 into Shared Memory
Thread 3 copied 4 into Shared Memory
Thread 4 copied 5 into Shared Memory
Thread 5 copied 6 into Shared Memory
Thread 6 copied 7 into Shared Memory
Thread 7 copied 8 into Shared Memory

Stride 4

Stride 4

Stride 4

Stride 4
Thread 0 : 1 + 5 = Thread 1 : 2 + 6 = Thread 2 : 3 + 7 = Thread 3 : 4 + 8 = 6
8
10
12

Shared Memory after Stride 4 : 6 8 10 12 
------------------------------------

Stride 2

Stride 2
Thread 0 : 6 + 10 = Thread 1 : 8 + 12 = 16
20

Shared Memory after Stride 2 : 16 20 
------------------------------------

Stride 1
Thread 0 : 16 + 20 = 36

Shared Memory after Stride 1 : 36 
------------------------------------

Reduction Completed.
Final 

---
<div align="center">


## EXPERIMENT 3

## PREFIX SUMMATION (SCAN)

</div>

---

## Prefix Summation (Scan)

**Prefix Summation**, also known as **Prefix Scan** or simply **Scan**, is a parallel algorithm used to compute the cumulative sum of elements in an array. Each output element contains the sum of all previous elements, including or excluding the current element, depending on the type of scan.

In CUDA, multiple GPU threads work together to compute prefix sums in parallel, significantly reducing execution time for large datasets compared to a sequential CPU implementation.

There are two types of Prefix Summation:

1. **Inclusive Scan** – Includes the current element in the cumulative sum.
2. **Exclusive Scan** – Excludes the current element from the cumulative sum.

This experiment demonstrates the **Inclusive Prefix Sum**.


### Applications of Prefix Summation

Prefix Summation is one of the most important parallel algorithms used in GPU computing. It serves as a building block for many advanced algorithms and applications, including:

- Parallel Sorting Algorithms
- Histogram Computation
- Image Processing
- Graph Algorithms
- Stream Compaction
- Sparse Matrix Operations
- Machine Learning
- Scientific Computing

By performing cumulative computations in parallel, Prefix Summation improves the efficiency and scalability of many GPU-based applications.

##  1 Inclusive Prefix Summation (Inclusive Scan)

**Inclusive Prefix Summation (Inclusive Scan)** is a parallel algorithm that computes the cumulative sum of an array. In an Inclusive Scan, each output element contains the sum of the current element and all the preceding elements in the input array, including the current element itself.

Inclusive Scan is widely used in parallel computing for applications such as data processing, image processing, sorting algorithms, graph algorithms, and scientific computing. In CUDA, it is efficiently implemented using multiple GPU threads, shared memory, and thread synchronization to perform the computation in parallel.

##  3.1.A Basic Implementation (Inclusive Scan)

In [ ]:
%%writefile inclusive_scan.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

__global__ void inclusiveScan(int *A, int *B);

int main()
{
    int h_A[N] = {1,2,3,4,5,6,7,8};
    int h_B[N];

    int *d_A, *d_B;

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));

    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);

    inclusiveScan<<<1, N>>>(d_A, d_B);

    cudaDeviceSynchronize();

    cudaMemcpy(h_B, d_B, N * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Inclusive Prefix Sum:\n");

    for(int i = 0; i < N; i++)
    {
        printf("%d ", h_B[i]);
    }

    printf("\n");

    cudaFree(d_A);
    cudaFree(d_B);

    return 0;
}

__global__ void inclusiveScan(int *A, int *B)
{
    int idx = threadIdx.x;

    if(idx < N)
    {
        int sum = 0;

        for(int i = 0; i <= idx; i++)
        {
            sum += A[i];
        }

        B[idx] = sum;
    }
}

Writing inclusive_scan.cu


In [ ]:
!nvcc inclusive_scan.cu -o inclusive_scan

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./inclusive_scan

Inclusive Prefix Sum:
1 3 6 10 15 21 28 36 


## Program 3.1.B: Detailed Implementation

In [ ]:
%%writefile inclusive_scan_detail.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

__global__ void inclusiveScan(int *A, int *B);

int main()
{
    // Host Arrays
    int h_A[N] = {1,2,3,4,5,6,7,8};
    int h_B[N];

    // Device Pointers
    int *d_A, *d_B;

    printf("=========================================\n");
    printf(" Inclusive Prefix Summation Using CUDA\n");
    printf("=========================================\n\n");

    printf("Input Array:\n");
    for(int i=0;i<N;i++)
        printf("%d ", h_A[i]);

    printf("\n\nAllocating GPU Memory...\n");

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));

    printf("GPU Memory Allocated Successfully.\n");

    printf("\nCopying Input Array from CPU to GPU...\n");

    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);

    printf("Data Copied Successfully.\n");

    printf("\nLaunching CUDA Kernel...\n\n");

    inclusiveScan<<<1, N>>>(d_A, d_B);

    cudaDeviceSynchronize();

    printf("\nKernel Execution Completed.\n");

    printf("\nCopying Result from GPU to CPU...\n");

    cudaMemcpy(h_B, d_B, N * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Result Copied Successfully.\n");

    printf("\nFinal Inclusive Prefix Sum:\n");

    for(int i=0;i<N;i++)
        printf("%d ", h_B[i]);

    printf("\n");

    printf("\nReleasing GPU Memory...\n");

    cudaFree(d_A);
    cudaFree(d_B);

    printf("GPU Memory Released.\n");

    printf("\nProgram Completed Successfully.\n");

    return 0;
}

// CUDA Kernel
__global__ void inclusiveScan(int *A, int *B)
{
    int idx = threadIdx.x;

    if(idx < N)
    {
        int sum = 0;

        printf("Thread %d started.\n", idx);

        for(int i=0;i<=idx;i++)
        {
            sum += A[i];

            printf("Thread %d : Adding A[%d]=%d  Current Sum=%d\n",
                   idx, i, A[i], sum);
        }

        B[idx] = sum;

        printf("Thread %d : Stored %d in Output[%d]\n\n",
               idx, B[idx], idx);
    }
}

Writing inclusive_scan_detail.cu


In [ ]:
!nvcc inclusive_scan_detail.cu -o inclusive_scan_detail

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./inclusive_scan_detail

 Inclusive Prefix Summation Using CUDA

Input Array:
1 2 3 4 5 6 7 8 

Allocating GPU Memory...
GPU Memory Allocated Successfully.

Copying Input Array from CPU to GPU...
Data Copied Successfully.

Launching CUDA Kernel...

Thread 0 started.
Thread 1 started.
Thread 2 started.
Thread 3 started.
Thread 4 started.
Thread 5 started.
Thread 6 started.
Thread 7 started.
Thread 3 : Adding A[0]=1  Current Sum=1
Thread 4 : Adding A[0]=1  Current Sum=1
Thread 5 : Adding A[0]=1  Current Sum=1
Thread 6 : Adding A[0]=1  Current Sum=1
Thread 7 : Adding A[0]=1  Current Sum=1
Thread 3 : Adding A[1]=2  Current Sum=3
Thread 4 : Adding A[1]=2  Current Sum=3
Thread 5 : Adding A[1]=2  Current Sum=3
Thread 6 : Adding A[1]=2  Current Sum=3
Thread 7 : Adding A[1]=2  Current Sum=3
Thread 3 : Adding A[2]=3  Current Sum=6
Thread 4 : Adding A[2]=3  Current Sum=6
Thread 5 : Adding A[2]=3  Current Sum=6
Thread 6 : Adding A[2]=3  Current Sum=6
Thread 7 : Adding A[2]=3  Current Sum=6
Thread 3 : Adding A[3]=4  Curren

## 3.2 Exclusive Prefix Summation (Exclusive Scan)

**Exclusive Prefix Summation (Exclusive Scan)** is a process of computing the cumulative sum of an array. In an Exclusive Scan, each output element is the sum of all the previous elements in the input array, excluding the current element. The first output element is always **0** because there are no elements before the first element.

In CUDA, each GPU thread computes one element of the output array. Each thread adds all the elements before its own index and stores the cumulative sum in the output array. This is the simplest implementation of Exclusive Scan and is useful for understanding the basic concept before learning optimized parallel scan algorithms.

## 3.2.A Basic Implementation (Exclusive Scan)

In [ ]:
%%writefile basic_exclusive_scan.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

// Function Prototype
__global__ void exclusiveScan(int *A, int *B);

int main()
{
    // Host Arrays
    int h_A[N] = {1,2,3,4,5,6,7,8};
    int h_B[N];

    // Device Pointers
    int *d_A, *d_B;

    // Allocate GPU Memory
    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));

    // Copy Input Array to GPU
    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);

    // Launch CUDA Kernel
    exclusiveScan<<<1, N>>>(d_A, d_B);

    // Wait for GPU to Finish
    cudaDeviceSynchronize();

    // Copy Result Back to CPU
    cudaMemcpy(h_B, d_B, N * sizeof(int), cudaMemcpyDeviceToHost);

    // Display Result
    printf("Exclusive Prefix Sum:\n");

    for(int i = 0; i < N; i++)
    {
        printf("%d ", h_B[i]);
    }

    printf("\n");

    // Free GPU Memory
    cudaFree(d_A);
    cudaFree(d_B);

    return 0;
}

// CUDA Kernel
__global__ void exclusiveScan(int *A, int *B)
{
    int idx = threadIdx.x;

    if(idx < N)
    {
        int sum = 0;

        for(int i = 0; i < idx; i++)
        {
            sum += A[i];
        }

        B[idx] = sum;
    }
}

Writing basic_exclusive_scan.cu


In [ ]:
!nvcc basic_exclusive_scan.cu -o basic_exclusive_scan

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./basic_exclusive_scan

Exclusive Prefix Sum:
0 1 3 6 10 15 21 28 


## Program 3.2.B: Detailed Implementation

In [ ]:
%%writefile detailed_exclusive_scan_detail.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

// Function Prototype
__global__ void exclusiveScan(int *A, int *B);

int main()
{
    // Host Arrays
    int h_A[N] = {1,2,3,4,5,6,7,8};
    int h_B[N];

    // Device Pointers
    int *d_A, *d_B;

    printf("=========================================\n");
    printf(" Exclusive Prefix Summation Using CUDA\n");
    printf("=========================================\n\n");

    printf("Input Array:\n");
    for(int i = 0; i < N; i++)
        printf("%d ", h_A[i]);

    printf("\n\nAllocating GPU Memory...\n");

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));

    printf("GPU Memory Allocated Successfully.\n");

    printf("\nCopying Input Array from CPU to GPU...\n");

    cudaMemcpy(d_A, h_A, N * sizeof(int), cudaMemcpyHostToDevice);

    printf("Data Copied Successfully.\n");

    printf("\nLaunching CUDA Kernel...\n\n");

    exclusiveScan<<<1, N>>>(d_A, d_B);

    cudaDeviceSynchronize();

    printf("\nKernel Execution Completed.\n");

    printf("\nCopying Result from GPU to CPU...\n");

    cudaMemcpy(h_B, d_B, N * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Result Copied Successfully.\n");

    printf("\nFinal Exclusive Prefix Sum:\n");

    for(int i = 0; i < N; i++)
        printf("%d ", h_B[i]);

    printf("\n");

    printf("\nReleasing GPU Memory...\n");

    cudaFree(d_A);
    cudaFree(d_B);

    printf("GPU Memory Released.\n");

    printf("\nProgram Completed Successfully.\n");

    return 0;
}

// CUDA Kernel
__global__ void exclusiveScan(int *A, int *B)
{
    int idx = threadIdx.x;

    if(idx < N)
    {
        int sum = 0;

        printf("Thread %d started.\n", idx);

        if(idx == 0)
        {
            printf("Thread 0 : No previous elements.\n");
            B[0] = 0;
            printf("Thread 0 : Stored 0 in Output[0]\n\n");
        }
        else
        {
            for(int i = 0; i < idx; i++)
            {
                sum += A[i];

                printf("Thread %d : Adding A[%d] = %d   Current Sum = %d\n",
                       idx, i, A[i], sum);
            }

            B[idx] = sum;

            printf("Thread %d : Stored %d in Output[%d]\n\n",
                   idx, B[idx], idx);
        }
    }
}

Writing detailed_exclusive_scan_detail.cu


In [ ]:
!nvcc detailed_exclusive_scan_detail.cu -o detailed_exclusive_scan_detail

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


The core difference lies in whether an element includes itself in its output result. Here's the breakdown:

Inclusive Scan: The output at position i is the sum (or result of the operation) of all elements from the start up to and including i.

Exclusive Scan: The output at position i is the sum of all elements before i. The first element is always the identity value (e.g., 0 for addition).

🔢 Example with Numbers
To make it concrete, let's apply a plus (+) scan to the input array [8, 6, 7].

Input Element	Inclusive Scan Output (sum up to i)	Exclusive Scan Output (sum before i)
8 (i=0)	8	0 (the identity)
6 (i=1)	8 + 6 = 14	8
7 (i=2)	8 + 6 + 7 = 21	8 + 6 = 14
An inclusive scan produces a new array where each element is the running total up to that point. In contrast, the exclusive scan shifts the results right by one element, inserting the identity value (like 0 for sum) at the beginning.

⚙️ Why This Distinction Matters in CUDA
In CUDA programming, this isn't just an academic difference; it has practical implications:

Exclusive is Key for Algorithms: Exclusive scans are the fundamental building block for many parallel algorithms, such as stream compaction (filtering out unwanted elements from an array). The exclusive scan creates unique, non-overlapping output indices for each element that passes the filter, which is essential for efficiently writing results in parallel without conflict.

They Can Convert Easily: An inclusive scan can be turned into an exclusive one, and vice-versa, with a simple, fast shift operation. A common approach is to implement an inclusive scan first and then convert it to an exclusive one in a separate, simple step. The CUDA CUB library provides dedicated, highly optimized functions for both through cub::DeviceScan::InclusiveSum and cub::DeviceScan::ExclusiveSum.

Would you like to know more about the standard parallel algorithms (like the Blelloch scan) used to implement these operations on the GPU?

In [ ]:
!./detailed_exclusive_scan_detail

 Exclusive Prefix Summation Using CUDA

Input Array:
1 2 3 4 5 6 7 8 

Allocating GPU Memory...
GPU Memory Allocated Successfully.

Copying Input Array from CPU to GPU...
Data Copied Successfully.

Launching CUDA Kernel...

Thread 0 started.
Thread 1 started.
Thread 2 started.
Thread 3 started.
Thread 4 started.
Thread 5 started.
Thread 6 started.
Thread 7 started.
Thread 0 : No previous elements.
Thread 0 : Stored 0 in Output[0]

Thread 4 : Adding A[0] = 1   Current Sum = 1
Thread 5 : Adding A[0] = 1   Current Sum = 1
Thread 6 : Adding A[0] = 1   Current Sum = 1
Thread 7 : Adding A[0] = 1   Current Sum = 1
Thread 4 : Adding A[1] = 2   Current Sum = 3
Thread 5 : Adding A[1] = 2   Current Sum = 3
Thread 6 : Adding A[1] = 2   Current Sum = 3
Thread 7 : Adding A[1] = 2   Current Sum = 3
Thread 4 : Adding A[2] = 3   Current Sum = 6
Thread 5 : Adding A[2] = 3   Current Sum = 6
Thread 6 : Adding A[2] = 3   Current Sum = 6
Thread 7 : Adding A[2] = 3   Current Sum = 6
Thread 4 : Adding A[3] = 